# Phase 2c: Training LightGBM with ADASYN

This notebook tests the **Adaptive Synthetic Sampling (ADASYN)** technique. After observing that SMOTE failed to improve the classification of our rarest classes (`H3` and `H4`), ADASYN is our next experiment. It is designed to be more effective by generating more synthetic data for the samples that are harder to learn.

**Goal:** Quickly determine if ADASYN can improve recall for the `H3` and `H4` classes where SMOTE failed. This will help us decide whether to continue with algorithmic improvements or focus on data acquisition.

**Steps:**
1. Load the pre-processed dataset.
2. Split data into training and testing sets.
3. **Apply ADASYN *only* to the training data**.
4. Train a `LGBMClassifier` on the ADASYN-balanced data.
5. Evaluate the model and compare its performance to the SMOTE model.

### 1. Import Necessary Libraries

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import os

from sklearn.impute import SimpleImputer


# The key import for ADASYN
from imblearn.over_sampling import ADASYN

print("Libraries imported successfully.")

### 2. Load and Prepare the Dataset

In [ ]:
DATASET_PATH = './AIH_model/AIH_preprocessor/final_training_dataset.csv'
df = pd.read_csv(DATASET_PATH)

X = df.drop('label', axis=1)
y = df['label']

# Define the label mapping for later visualization
label_map = {
    0: "BODY_TEXT", 1: "TITLE", 2: "H1",
    3: "H2", 4: "H3", 5: "H4"
}
label_names = [label_map[i] for i in sorted(y.unique())]

print("✅ Dataset loaded and prepared.")
print(f"X shape: {X.shape}, y shape: {y.shape}")

### 3. Split Data into Training and Testing Sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print("Data split into training and testing sets.")

### 4. Apply ADASYN to the Training Data

This is the key change from the previous notebook. We replace `SMOTE` with `ADASYN`. Just like with SMOTE, we must adjust the `n_neighbors` parameter to be less than the number of samples in our smallest class to prevent the algorithm from crashing.

In [ ]:
print("Class distribution before ADASYN:")
print(y_train.value_counts().sort_index())
print(f"\nChecking for NaN values in X_train...")
nan_count = X_train.isnull().sum().sum()
print(f"Found {nan_count} NaN values in the training data.")

if nan_count > 0:
    print("Applying imputation to fill missing values...")
    # Initialize the imputer to replace NaNs with the median of each column
    imputer = SimpleImputer(strategy='median')
    
    # Fit the imputer on the training data and transform it
    # We use .fit_transform on the training data
    X_train_imputed = imputer.fit_transform(X_train)
    # IMPORTANT: We only use .transform on the test data to avoid data leakage
    X_test_imputed = imputer.transform(X_test)
    
    # Convert the imputed numpy arrays back to pandas DataFrames
    X_train = pd.DataFrame(X_train_imputed, columns=X.columns)
    X_test = pd.DataFrame(X_test_imputed, columns=X.columns)
    
    print("✅ Missing values handled.")
else:
    print("✅ No missing values found.")
# Find the number of samples in the smallest class to set n_neighbors
min_samples = y_train.value_counts().min()
n = min_samples - 1 if min_samples > 1 else 1
print(f"\nSmallest class has {min_samples} samples. Setting n_neighbors for ADASYN to {n}.\n")

# Initialize ADASYN
adasyn = ADASYN(random_state=42, n_neighbors=n)

# Fit and apply ADASYN
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

print("Class distribution after ADASYN:")
print(y_train_resampled.value_counts().sort_index())

print(f"Original training set size: {len(X_train)}")
print(f"Resampled training set size: {len(X_train_resampled)}")

### 5. Train the LightGBM Model on ADASYN-Resampled Data

In [ ]:
lgbm_adasyn = lgb.LGBMClassifier(random_state=42)

print("Training the LightGBM model on ADASYN-balanced data...")
lgbm_adasyn.fit(X_train_resampled, y_train_resampled)
print("✅ Model training complete.")

### 6. Evaluate Model Performance on the Original Test Set

In [ ]:
y_pred_adasyn = lgbm_adasyn.predict(X_test)

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_adasyn):.4f}\n")

print("Classification Report (with ADASYN):")
print(classification_report(y_test, y_pred_adasyn, target_names=label_names))

print("Confusion Matrix (with ADASYN):")
cm = confusion_matrix(y_test, y_pred_adasyn)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix for LightGBM with ADASYN')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

### 7. Verdict: Is It Worthwhile?

After running this notebook, compare the final **Classification Report** with the one from the SMOTE experiment.

- **If `recall` for `H3` and `H4` is now greater than 0:** ADASYN has proven to be a superior technique for your dataset. This is a strong signal that continuing with algorithmic tuning is a worthwhile path.
- **If `recall` for `H3` and `H4` is still 0:** This suggests that the feature patterns for these classes are so subtle or indistinct in your current data that even advanced resampling cannot help the model learn them. In this case, the problem is not with the algorithm, but with the data itself. Your time would be better spent acquiring and labeling new documents from sources like DocBank or DocLayNet to provide the model with more examples of these rare classes.